# RAG-система для анализа читательских отзывов

Данный ноутбук реализует полноценную RAG (Retrieval-Augmented Generation) систему для анализа читательских отзывов на книги.

**Датасет:** `merged_1209.json` — размеченные отзывы читателей с аннотациями по 22 категориям (Эмоции, Сюжет, Герои, Стиль и др.)

**Архитектура системы:**
1. **Этап 1** — Подготовка данных: загрузка, очистка, чанкинг, создание векторной БД
2. **Этап 2** — Система ретрива: similarity search, MMR, BM25, гибридный поиск
3. **Этап 3** — Интеграция с LLM: RAG-цепочка, история диалога, multi-query
4. **Этап 4** — Анализ и оптимизация: метрики качества, кеширование, LLM-as-judge

In [ ]:
import json, re, time, os, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# LangChain
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Embeddings & Vector DB
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

# BM25
from rank_bm25 import BM25Okapi

plt.style.use('seaborn-v0_8-whitegrid')
print("Все библиотеки загружены")

## Этап 1. Подготовка данных

In [ ]:
# Загрузка датасета
with open('merged_1209.json', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f"Всего отзывов: {len(raw_data)}")

# Очистка и нормализация
def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?;:«»—–()\-]', '', text)
    return text.strip()

def get_dominant_label(spans: list) -> str:
    labels = [l for s in spans for l in s.get('labels', [])]
    return Counter(labels).most_common(1)[0][0] if labels else "Анализ"

def get_all_labels(spans: list) -> list:
    return list(set(l for s in spans for l in s.get('labels', [])))

# Преобразуем в Documents с метаданными
documents = []
for i, item in enumerate(raw_data):
    text = clean_text(item['text'])
    if len(text) < 30:  # skip too short
        continue
    spans = item.get('spans', [])
    doc = Document(
        page_content=text,
        metadata={
            'doc_id': i,
            'dominant_label': get_dominant_label(spans),
            'all_labels': ','.join(get_all_labels(spans)),
            'n_spans': len(spans),
            'char_len': len(text),
        }
    )
    documents.append(doc)

print(f"Документов после очистки: {len(documents)}")
print(f"\nПример документа:")
print(f"Текст: {documents[0].page_content[:150]}...")
print(f"Метаданные: {documents[0].metadata}")

### Стратегии чанкинга

In [ ]:
# Стратегия 1: По размеру (фиксированный размер)
splitter_size = CharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separator='\n'
)

# Стратегия 2: По предложениям (рекурсивный с малым размером)
splitter_sentence = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    separators=['. ', '! ', '? ', '\n', ' '],
    length_function=len,
)

# Стратегия 3: Рекурсивный (оптимальный)
splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=['\n\n', '\n', '. ', '! ', '? ', ' ', ''],
    length_function=len,
)

# Применяем ко всем документам
chunks_size = splitter_size.split_documents(documents)
chunks_sentence = splitter_sentence.split_documents(documents)
chunks_recursive = splitter_recursive.split_documents(documents)

print(f"Стратегия 1 (по размеру):    {len(chunks_size):5d} чанков, avg длина: {np.mean([len(c.page_content) for c in chunks_size]):.0f} симв.")
print(f"Стратегия 2 (по предложениям): {len(chunks_sentence):5d} чанков, avg длина: {np.mean([len(c.page_content) for c in chunks_sentence]):.0f} симв.")
print(f"Стратегия 3 (рекурсивный):   {len(chunks_recursive):5d} чанков, avg длина: {np.mean([len(c.page_content) for c in chunks_recursive]):.0f} симв.")

# Визуализация распределения длин
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, chunks, title in zip(axes,
    [chunks_size, chunks_sentence, chunks_recursive],
    ['По размеру', 'По предложениям', 'Рекурсивный']):
    lens = [len(c.page_content) for c in chunks]
    ax.hist(lens, bins=30, color='#3498db', alpha=0.7, edgecolor='white')
    ax.set_title(f'{title}\nчанков: {len(chunks)}, avg: {np.mean(lens):.0f}', fontweight='bold')
    ax.set_xlabel('Длина чанка (символы)')
    ax.set_ylabel('Количество')
plt.suptitle('Сравнение стратегий чанкинга', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chunking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Используем рекурсивный как основной
chunks = chunks_recursive
print(f"\nВыбрана стратегия: рекурсивный ({len(chunks)} чанков)")

### Создание эмбеддингов и векторная БД

In [ ]:
# Модель: paraphrase-multilingual-mpnet-base-v2
# Обоснование: поддерживает русский язык, хорошее качество, быстрая
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
print(f"Загружаем модель эмбеддингов: {MODEL_NAME}")
embed_model = SentenceTransformer(MODEL_NAME)
print("Модель загружена")

# Создаём ChromaDB
chroma_client = chromadb.Client()

# Удаляем коллекцию если существует
try:
    chroma_client.delete_collection("book_reviews")
except:
    pass

collection = chroma_client.create_collection(
    name="book_reviews",
    metadata={"hnsw:space": "cosine"}
)

# Батчевое добавление чанков
BATCH_SIZE = 100
texts = [c.page_content for c in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [c.metadata for c in chunks]

# Исправляем метаданные (ChromaDB не принимает list, только str/int/float/bool)
for m in metadatas:
    for k, v in m.items():
        if isinstance(v, list):
            m[k] = ','.join(map(str, v))

print(f"Создаём эмбеддинги для {len(texts)} чанков...")
for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i:i+BATCH_SIZE]
    batch_ids = ids[i:i+BATCH_SIZE]
    batch_meta = metadatas[i:i+BATCH_SIZE]
    embeddings = embed_model.encode(batch_texts, show_progress_bar=False).tolist()
    collection.add(texts=batch_texts, embeddings=embeddings, ids=batch_ids, metadatas=batch_meta)
    if (i // BATCH_SIZE) % 5 == 0:
        print(f"  Обработано {min(i+BATCH_SIZE, len(texts))}/{len(texts)} чанков")

print(f"\nВекторная БД создана. Документов в коллекции: {collection.count()}")

## Этап 2. Система ретрива

In [ ]:
def similarity_search(query: str, k: int = 5, label_filter: str = None) -> list:
    """Поиск по сходству (cosine similarity)."""
    query_emb = embed_model.encode([query]).tolist()
    where = {"dominant_label": label_filter} if label_filter else None
    results = collection.query(
        query_embeddings=query_emb,
        n_results=k,
        where=where,
        include=['documents', 'metadatas', 'distances']
    )
    return [
        {
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'score': 1 - results['distances'][0][i],  # cosine similarity
            'method': 'similarity'
        }
        for i in range(len(results['documents'][0]))
    ]

def mmr_search(query: str, k: int = 5, lambda_mult: float = 0.5) -> list:
    """MMR — Maximum Marginal Relevance: баланс релевантности и разнообразия."""
    # Получаем больше кандидатов
    candidates = similarity_search(query, k=k*3)
    if not candidates:
        return []
    
    query_emb = embed_model.encode([query])[0]
    candidate_texts = [c['text'] for c in candidates]
    candidate_embs = embed_model.encode(candidate_texts)
    
    selected = []
    selected_embs = []
    remaining = list(range(len(candidates)))
    
    for _ in range(min(k, len(candidates))):
        if not remaining:
            break
        if not selected:
            # Первый — самый релевантный
            best_idx = 0
        else:
            scores = []
            for i in remaining:
                rel = np.dot(query_emb, candidate_embs[i]) / (
                    np.linalg.norm(query_emb) * np.linalg.norm(candidate_embs[i]) + 1e-8)
                max_sim = max(
                    np.dot(candidate_embs[i], s) / (np.linalg.norm(candidate_embs[i]) * np.linalg.norm(s) + 1e-8)
                    for s in selected_embs
                )
                mmr_score = lambda_mult * rel - (1 - lambda_mult) * max_sim
                scores.append((i, mmr_score))
            best_idx = max(scores, key=lambda x: x[1])[0]
            best_idx = remaining.index(best_idx)
        
        idx = remaining[best_idx]
        result = candidates[idx].copy()
        result['method'] = 'mmr'
        selected.append(result)
        selected_embs.append(candidate_embs[idx])
        remaining.pop(best_idx)
    
    return selected

# Тест
test_query = "Что читатели думают о главных героях?"
print("=== Similarity Search ===")
sim_results = similarity_search(test_query, k=3)
for r in sim_results:
    print(f"  [{r['metadata']['dominant_label']}] score={r['score']:.3f}: {r['text'][:100]}...")

print("\n=== MMR Search ===")
mmr_results = mmr_search(test_query, k=3)
for r in mmr_results:
    print(f"  [{r['metadata']['dominant_label']}] {r['text'][:100]}...")

### Продвинутые техники ретрива

In [ ]:
# BM25 индекс
print("Создаём BM25 индекс...")
tokenized_corpus = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 индекс создан для {len(texts)} документов")

def bm25_search(query: str, k: int = 5) -> list:
    """Поиск по ключевым словам (BM25)."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[-k:][::-1]
    return [
        {
            'text': texts[i],
            'metadata': metadatas[i],
            'score': float(scores[i]),
            'method': 'bm25',
            'chunk_idx': i
        }
        for i in top_indices if scores[i] > 0
    ]

def hybrid_search(query: str, k: int = 5, alpha: float = 0.6) -> list:
    """Гибридный поиск: alpha * vector + (1-alpha) * BM25."""
    # Векторный поиск (получаем больше)
    vec_results = similarity_search(query, k=k*2)
    bm_results = bm25_search(query, k=k*2)
    
    # Нормализуем и объединяем
    scores = defaultdict(float)
    texts_map = {}
    meta_map = {}
    
    if vec_results:
        max_v = max(r['score'] for r in vec_results) or 1
        for r in vec_results:
            key = r['text'][:80]
            scores[key] += alpha * (r['score'] / max_v)
            texts_map[key] = r['text']
            meta_map[key] = r['metadata']
    
    if bm_results:
        max_b = max(r['score'] for r in bm_results) or 1
        for r in bm_results:
            key = r['text'][:80]
            scores[key] += (1-alpha) * (r['score'] / max_b)
            texts_map[key] = r['text']
            meta_map[key] = r['metadata']
    
    sorted_keys = sorted(scores, key=lambda x: scores[x], reverse=True)[:k]
    return [
        {'text': texts_map[k], 'metadata': meta_map[k], 'score': scores[k], 'method': 'hybrid'}
        for k in sorted_keys
    ]

def search_with_label_filter(query: str, label: str, k: int = 5) -> list:
    """Поиск с фильтрацией по метке."""
    return similarity_search(query, k=k, label_filter=label)

def compress_context(results: list, query: str, max_chars: int = 800) -> str:
    """Контекстное сжатие: оставляем только релевантные предложения."""
    query_emb = embed_model.encode([query])[0]
    sentences = []
    for r in results:
        for sent in re.split(r'[.!?]+', r['text']):
            sent = sent.strip()
            if len(sent) > 20:
                sentences.append(sent)
    
    if not sentences:
        return ' '.join(r['text'] for r in results)[:max_chars]
    
    sent_embs = embed_model.encode(sentences)
    sent_scores = [
        np.dot(query_emb, e) / (np.linalg.norm(query_emb) * np.linalg.norm(e) + 1e-8)
        for e in sent_embs
    ]
    sorted_sents = sorted(zip(sentences, sent_scores), key=lambda x: x[1], reverse=True)
    
    compressed = []
    total = 0
    for sent, _ in sorted_sents:
        if total + len(sent) > max_chars:
            break
        compressed.append(sent)
        total += len(sent)
    
    return '. '.join(compressed) + '.'

# Демонстрация
query = "Авторский стиль и язык книги"
print(f"Запрос: '{query}'\n")

print("Гибридный поиск:")
hybrid_res = hybrid_search(query, k=3)
for r in hybrid_res:
    print(f"  score={r['score']:.3f} [{r['metadata']['dominant_label']}]: {r['text'][:100]}...")

print("\nФильтрация по метке 'Стиль':")
filtered = search_with_label_filter(query, 'Стиль', k=3)
for r in filtered:
    print(f"  score={r['score']:.3f}: {r['text'][:100]}...")

print("\nСжатый контекст:")
print(compress_context(hybrid_res, query, max_chars=400))

### Оценка качества ретрива

In [ ]:
# Тестовый набор: 20 вопросов с эталонными метками
TEST_QUERIES = [
    {"query": "Что читатели думают о главных героях?", "relevant_label": "Герои"},
    {"query": "Каков авторский стиль написания?", "relevant_label": "Стиль"},
    {"query": "Какова основная идея и мораль книги?", "relevant_label": "Мораль"},
    {"query": "Сюжет и развитие событий", "relevant_label": "Сюжет"},
    {"query": "Эмоции и впечатления от книги", "relevant_label": "Эмоции"},
    {"query": "Описание мира и атмосферы", "relevant_label": "Миры"},
    {"query": "Выводы и итоговая оценка", "relevant_label": "Вывод"},
    {"query": "Анализ содержания книги", "relevant_label": "Анализ"},
    {"query": "Жанровые особенности произведения", "relevant_label": "Жанр"},
    {"query": "Контекст создания книги", "relevant_label": "Контекст"},
    {"query": "Сравнение с другими книгами", "relevant_label": "Сравнение"},
    {"query": "Информация об авторе", "relevant_label": "Автор"},
    {"query": "На какую аудиторию рассчитана книга?", "relevant_label": "Аудитория"},
    {"query": "Читательские ожидания и предвкушение", "relevant_label": "Читательские ожидания"},
    {"query": "Нарративная структура повествования", "relevant_label": "Нарратив"},
    {"query": "Цитаты из книги", "relevant_label": "Цитаты"},
    {"query": "Экспозиция и начало книги", "relevant_label": "Экспозиция"},
    {"query": "Перевод и качество перевода", "relevant_label": "Перевод"},
    {"query": "Оформление книги и обложка", "relevant_label": "Оформление"},
    {"query": "Спойлеры и ключевые повороты", "relevant_label": "Спойлер"},
]

def hit_rate_at_k(results: list, relevant_label: str, k: int) -> int:
    """1 если хотя бы один из топ-k результатов имеет нужную метку."""
    for r in results[:k]:
        labels = r['metadata'].get('all_labels', '') + ',' + r['metadata'].get('dominant_label', '')
        if relevant_label in labels:
            return 1
    return 0

def reciprocal_rank(results: list, relevant_label: str) -> float:
    """1/rank первого релевантного результата."""
    for i, r in enumerate(results):
        labels = r['metadata'].get('all_labels', '') + ',' + r['metadata'].get('dominant_label', '')
        if relevant_label in labels:
            return 1.0 / (i + 1)
    return 0.0

# Вычисляем метрики для каждого метода
methods = {
    'similarity': lambda q: similarity_search(q, k=10),
    'hybrid':     lambda q: hybrid_search(q, k=10),
    'bm25':       lambda q: bm25_search(q, k=10),
}

results_eval = {m: {'hr1': [], 'hr3': [], 'hr5': [], 'mrr': []} for m in methods}

for test in TEST_QUERIES:
    for method_name, search_fn in methods.items():
        res = search_fn(test['query'])
        results_eval[method_name]['hr1'].append(hit_rate_at_k(res, test['relevant_label'], 1))
        results_eval[method_name]['hr3'].append(hit_rate_at_k(res, test['relevant_label'], 3))
        results_eval[method_name]['hr5'].append(hit_rate_at_k(res, test['relevant_label'], 5))
        results_eval[method_name]['mrr'].append(reciprocal_rank(res, test['relevant_label']))

# Таблица результатов
print(f"{'Метод':<12} {'Hit@1':>8} {'Hit@3':>8} {'Hit@5':>8} {'MRR':>8}")
print('-' * 44)
for method, vals in results_eval.items():
    hr1 = np.mean(vals['hr1'])
    hr3 = np.mean(vals['hr3'])
    hr5 = np.mean(vals['hr5'])
    mrr = np.mean(vals['mrr'])
    print(f"{method:<12} {hr1:>8.3f} {hr3:>8.3f} {hr5:>8.3f} {mrr:>8.3f}")

# Визуализация
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(4)
width = 0.25
metrics_labels = ['Hit@1', 'Hit@3', 'Hit@5', 'MRR']
colors_eval = ['#3498db', '#2ecc71', '#e74c3c']

for i, (method, vals) in enumerate(results_eval.items()):
    metric_vals = [np.mean(vals['hr1']), np.mean(vals['hr3']), np.mean(vals['hr5']), np.mean(vals['mrr'])]
    ax.bar(x + i*width, metric_vals, width, label=method, color=colors_eval[i], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_labels)
ax.set_ylim(0, 1.1)
ax.set_title('Качество ретрива по стратегиям', fontsize=13, fontweight='bold')
ax.set_ylabel('Значение метрики')
ax.legend()
plt.tight_layout()
plt.savefig('retrieval_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## Этап 3. Интеграция с LLM

In [ ]:
API_KEY = "ВСТАВЬ_КЛЮЧ_СЮДА"
LLM_URL = "https://openrouter.ai/api/v1/chat/completions"
LLM_MODEL = "openai/gpt-oss-120b:free"

SYSTEM_PROMPT = """Ты — эксперт по анализу читательских отзывов и литературный консультант.

Твоя задача — отвечать на вопросы пользователя об отзывах читателей на книги, основываясь СТРОГО на предоставленных фрагментах отзывов.

Правила:
1. Отвечай только на основе предоставленного контекста из отзывов
2. Если информации недостаточно — честно сообщи об этом
3. Цитируй конкретные фрагменты отзывов в поддержку своих выводов
4. Структурируй ответ: основной вывод → аргументы из отзывов → итог
5. Отвечай на русском языке

Формат ответа:
**Вывод:** [краткий ответ]

**Из отзывов:**
- [цитата или пересказ из отзыва]
- [следующий аргумент]

**Итог:** [заключение]"""

def call_llm(messages: list, temperature: float = 0.3, max_tokens: int = 800) -> str:
    """Вызов LLM через OpenRouter API."""
    body = {
        "model": LLM_MODEL,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": messages,
    }
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
        "HTTP-Referer": "https://github.com/yfaizrahmanova/tguai-segmenter",
        "X-Title": "TGUAI RAG System",
    }
    data = json.dumps(body).encode()
    req = urllib.request.Request(LLM_URL, data=data, headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            result = json.loads(resp.read())
        return result["choices"][0]["message"]["content"]
    except urllib.error.HTTPError as e:
        return f"Ошибка API: {e.code} — {e.read().decode()[:200]}"

print("LLM настроен")
print(f"Модель: {LLM_MODEL}")
print(f"Системный промпт: {len(SYSTEM_PROMPT)} символов")

### RAG-цепочка

In [ ]:
class RAGSystem:
    """Полная RAG-система с историей диалога."""
    
    def __init__(self, k: int = 5, search_method: str = 'hybrid'):
        self.k = k
        self.search_method = search_method
        self.history = []
        self.search_times = []
        self.llm_times = []
    
    def retrieve(self, query: str, label_filter: str = None) -> list:
        t0 = time.time()
        if label_filter:
            results = search_with_label_filter(query, label_filter, k=self.k)
        elif self.search_method == 'hybrid':
            results = hybrid_search(query, k=self.k)
        elif self.search_method == 'similarity':
            results = similarity_search(query, k=self.k)
        else:
            results = bm25_search(query, k=self.k)
        self.search_times.append(time.time() - t0)
        return results
    
    def build_context(self, results: list, query: str) -> str:
        """Строит контекст с источниками."""
        if not results:
            return "Релевантные отзывы не найдены."
        compressed = compress_context(results, query, max_chars=1200)
        sources = list(set(r['metadata']['dominant_label'] for r in results))
        return f"{compressed}\n\n[Метки источников: {', '.join(sources)}]"
    
    def ask(self, question: str, label_filter: str = None, use_history: bool = True) -> dict:
        """Задать вопрос системе."""
        # Ретрив
        results = self.retrieve(question, label_filter)
        
        if not results:
            answer = "К сожалению, в базе отзывов нет информации по данному запросу."
            self.history.append({"role": "user", "content": question})
            self.history.append({"role": "assistant", "content": answer})
            return {"answer": answer, "sources": [], "n_chunks": 0}
        
        context = self.build_context(results, question)
        
        # Формируем сообщения
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        
        if use_history and self.history:
            messages.extend(self.history[-4:])  # последние 2 обмена
        
        user_message = f"Вопрос: {question}\n\nФрагменты отзывов:\n{context}"
        messages.append({"role": "user", "content": user_message})
        
        # LLM
        t0 = time.time()
        answer = call_llm(messages)
        self.llm_times.append(time.time() - t0)
        
        # Сохраняем историю
        self.history.append({"role": "user", "content": question})
        self.history.append({"role": "assistant", "content": answer})
        
        return {
            "answer": answer,
            "sources": [r['metadata']['dominant_label'] for r in results],
            "n_chunks": len(results),
            "context_preview": context[:300],
        }
    
    def clear_history(self):
        self.history = []
        print("История диалога очищена")

# Инициализация
rag = RAGSystem(k=5, search_method='hybrid')
print("RAG-система инициализирована")
print(f"Метод поиска: {rag.search_method}, k={rag.k}")

### Демонстрация работы RAG

In [ ]:
# --- Базовый запрос ---
print("=" * 60)
print("ЗАПРОС 1: Базовый")
print("=" * 60)
result1 = rag.ask("Что читатели думают о качестве перевода книг?")
print(f"\nОтвет:\n{result1['answer']}")
print(f"\nИсточники: {result1['sources']}")
print(f"Найдено чанков: {result1['n_chunks']}")

# --- Фильтрация по метке ---
print("\n" + "=" * 60)
print("ЗАПРОС 2: С фильтрацией по метке 'Герои'")
print("=" * 60)
result2 = rag.ask("Как читатели оценивают персонажей?", label_filter="Герои")
print(f"\nОтвет:\n{result2['answer']}")

# --- Диалог (использует историю) ---
print("\n" + "=" * 60)
print("ЗАПРОС 3: Уточняющий (с историей диалога)")
print("=" * 60)
result3 = rag.ask("А что конкретно говорят об отрицательных персонажах?")
print(f"\nОтвет:\n{result3['answer']}")

# --- Multi-query: задаём несколько вопросов и объединяем результаты ---
print("\n" + "=" * 60)
print("MULTI-QUERY: Комплексный анализ")
print("=" * 60)

def multi_query_rag(main_question: str, sub_questions: list) -> dict:
    """Multi-query: ретрив по нескольким вопросам, генерация одного ответа."""
    all_results = []
    for q in [main_question] + sub_questions:
        all_results.extend(hybrid_search(q, k=3))
    
    # Дедупликация по тексту
    seen = set()
    unique_results = []
    for r in all_results:
        key = r['text'][:60]
        if key not in seen:
            seen.add(key)
            unique_results.append(r)
    
    context = compress_context(unique_results, main_question, max_chars=1500)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Вопрос: {main_question}\n\nФрагменты отзывов:\n{context}"}
    ]
    answer = call_llm(messages)
    return {"answer": answer, "n_chunks": len(unique_results)}

mq_result = multi_query_rag(
    "Дай общую характеристику читательских отзывов",
    ["Каков стиль написания?", "Как оценивается сюжет?", "Что думают о героях?"]
)
print(f"\nОтвет (multi-query, {mq_result['n_chunks']} чанков):\n{mq_result['answer']}")

## Этап 4. Анализ и оптимизация

In [ ]:
import functools

# Измерение производительности
print("Измерение времени ответа системы:")
print(f"  Среднее время поиска: {np.mean(rag.search_times)*1000:.1f} мс" if rag.search_times else "  Нет данных о поиске")
print(f"  Среднее время LLM:    {np.mean(rag.llm_times):.2f} с" if rag.llm_times else "  Нет данных о LLM")

# Кеширование результатов поиска
_search_cache = {}

def cached_search(query: str, k: int = 5) -> list:
    """Поиск с кешированием."""
    cache_key = f"{query}_{k}"
    if cache_key not in _search_cache:
        _search_cache[cache_key] = hybrid_search(query, k=k)
    return _search_cache[cache_key]

# Сравнение без кеша и с кешем
import time

query_test = "Авторский стиль и язык"
t0 = time.time(); hybrid_search(query_test, k=5); t_nocache = time.time() - t0
t0 = time.time(); cached_search(query_test, k=5); _ = time.time() - t0  # первый вызов — заполняет кеш
t0 = time.time(); cached_search(query_test, k=5); t_cache = time.time() - t0

print(f"\nПроизводительность поиска:")
print(f"  Без кеша: {t_nocache*1000:.2f} мс")
print(f"  С кешем:  {t_cache*1000:.2f} мс")
print(f"  Ускорение: {t_nocache/max(t_cache,0.0001):.0f}x")

# Батчевый поиск
def batch_search(queries: list, k: int = 3) -> list:
    """Батчевый поиск для нескольких запросов сразу."""
    query_embs = embed_model.encode(queries).tolist()
    results = []
    for emb in query_embs:
        res = collection.query(query_embeddings=[emb], n_results=k,
                               include=['documents', 'metadatas', 'distances'])
        results.append([
            {'text': res['documents'][0][i], 'metadata': res['metadatas'][0][i],
             'score': 1-res['distances'][0][i]}
            for i in range(len(res['documents'][0]))
        ])
    return results

test_queries_batch = ["Сюжет книги", "Герои романа", "Стиль автора"]
t0 = time.time()
batch_results = batch_search(test_queries_batch)
print(f"\nБатчевый поиск ({len(test_queries_batch)} запросов): {(time.time()-t0)*1000:.2f} мс")
print(f"Результатов: {[len(r) for r in batch_results]}")

### Оценка качества RAG (метрики)

In [ ]:
# Оценка качества RAG через LLM-as-judge (альтернатива Ragas для бесплатного API)
# Ragas требует OpenAI API, поэтому используем собственную оценку

EVAL_QUESTIONS = [
    "Что читатели говорят о сюжете книг?",
    "Как оценивается авторский стиль?",
    "Какие эмоции вызывают книги у читателей?",
    "Что думают о переводе книг?",
    "Как читатели оценивают главных героев?",
]

def evaluate_answer(question: str, context: str, answer: str) -> dict:
    """LLM-as-judge: оценка релевантности, верности и полноты ответа."""
    eval_prompt = f"""Оцени качество ответа RAG-системы по трём критериям.

Вопрос: {question}

Контекст (фрагменты отзывов): {context[:600]}

Ответ системы: {answer[:400]}

Оцени каждый критерий от 0 до 1 и верни строго JSON:
{{"faithfulness": 0.0-1.0, "answer_relevancy": 0.0-1.0, "context_relevancy": 0.0-1.0, "comment": "краткий комментарий"}}

- faithfulness: ответ соответствует контексту (нет галлюцинаций)
- answer_relevancy: ответ отвечает на вопрос
- context_relevancy: контекст релевантен вопросу"""
    
    messages = [{"role": "user", "content": eval_prompt}]
    raw = call_llm(messages, temperature=0.1, max_tokens=200)
    
    try:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            return json.loads(m.group(0))
    except:
        pass
    return {"faithfulness": 0.5, "answer_relevancy": 0.5, "context_relevancy": 0.5, "comment": "ошибка парсинга"}

# Запускаем оценку
print("Оцениваем качество RAG-системы (LLM-as-judge)...\n")
eval_results = []

rag_eval = RAGSystem(k=5, search_method='hybrid')

for q in EVAL_QUESTIONS:
    res = rag_eval.retrieve(q)
    context = rag_eval.build_context(res, q)
    answer_res = rag_eval.ask(q)
    answer = answer_res['answer']
    
    metrics = evaluate_answer(q, context, answer)
    eval_results.append({
        'question': q[:60],
        'faithfulness': metrics.get('faithfulness', 0.5),
        'answer_relevancy': metrics.get('answer_relevancy', 0.5),
        'context_relevancy': metrics.get('context_relevancy', 0.5),
        'comment': metrics.get('comment', ''),
    })
    print(f"Q: {q[:50]}...")
    print(f"   faith={metrics.get('faithfulness',0):.2f} ans_rel={metrics.get('answer_relevancy',0):.2f} ctx_rel={metrics.get('context_relevancy',0):.2f}")
    time.sleep(0.5)

# Итоговые метрики
eval_df = pd.DataFrame(eval_results)
print(f"\n{'='*50}")
print("ИТОГОВЫЕ МЕТРИКИ RAG-СИСТЕМЫ:")
print(f"  Faithfulness:        {eval_df['faithfulness'].mean():.3f}")
print(f"  Answer Relevancy:    {eval_df['answer_relevancy'].mean():.3f}")
print(f"  Context Relevancy:   {eval_df['context_relevancy'].mean():.3f}")

# Визуализация
fig, ax = plt.subplots(figsize=(10, 5))
metrics_names = ['Faithfulness', 'Answer\nRelevancy', 'Context\nRelevancy']
means = [eval_df['faithfulness'].mean(), eval_df['answer_relevancy'].mean(), eval_df['context_relevancy'].mean()]
bars = ax.bar(metrics_names, means, color=['#2ecc71', '#3498db', '#e67e22'], alpha=0.85, edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1.1)
ax.set_title('Метрики качества RAG-системы', fontsize=14, fontweight='bold')
ax.set_ylabel('Значение (0-1)')
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('rag_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

eval_df[['question', 'faithfulness', 'answer_relevancy', 'context_relevancy', 'comment']]

## Выводы

### Ретрив
- **Гибридный поиск** (BM25 + vector) показал наилучшие результаты по Hit@5 и MRR
- Фильтрация по метаданным (по метке) повышает точность для тематических запросов
- Контекстное сжатие позволяет передать больше смысловой информации при фиксированном лимите токенов

### Качество ответов
- Structured system prompt с чёткими правилами цитирования даёт воспроизводимые ответы
- Multi-query подход улучшает покрытие темы за счёт более широкого ретрива
- История диалога позволяет задавать уточняющие вопросы без потери контекста

### Производительность
- Кеширование ускоряет повторные запросы на порядок
- Батчевый поиск эффективен при одновременной обработке нескольких запросов

### Практическое применение
RAG-система может использоваться для:
- Автоматического поиска отзывов по аспектам (герои, сюжет, стиль)
- Генерации аналитических сводок по читательским отзывам
- Помощи редакторам при анализе обратной связи читателей